# 01 · Датасет
Открытые HF-датасеты + генераторы RU/EN атак, нормы и hard negatives → очистка → почти-дубли → разбиение → EDA.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess
REPO_DIR = '/content/prompt-injection-detector'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/HPAINH/prompt-injection-detector.git', REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '-q'], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
os.environ['PIG_ROOT'] = '/content/drive/MyDrive/prompt-injection'
os.makedirs(os.environ['PIG_ROOT'], exist_ok=True)

import logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s', force=True)
from pig.config import load_config, set_seed
cfg = load_config()
set_seed(cfg['seed'])
print('PIG_ROOT =', os.environ['PIG_ROOT'])

## 1.1–1.2 Открытые источники
EN: deepset, jackhhao, Lakera gandalf. RU: dmtrdr/russian_prompt_injections (атаки), IlyaGusev/ru_turbo_alpaca (норма). Строки Lakera из любых источников уходят в held-out.

In [ ]:
from pig.data.sources import load_hf_sources
hf = load_hf_sources(cfg['data']['max_per_hf_source'], cfg['seed'], cfg['data']['max_per_source_overrides'])
display(hf.groupby(['source', 'label', 'lang']).size().unstack(fill_value=0))
hf.sample(8, random_state=1)[['text', 'label', 'attack_type', 'source']]

## 1.3 Перевод части EN-атак (опционально, лучше на GPU)
Переводятся атаки из источников, кроме held-out источника. После перевода выгрузите лист проверки, проверьте ≥10% вручную.

In [ ]:
import torch
from pig.data.translate import translate_attacks, review_sheet
from pig.text import normalize
src = hf[~hf['source'].str.lower().str.contains('|'.join(cfg['data']['heldout_sources']))]
translated = translate_attacks(src, n=300, device=0 if torch.cuda.is_available() else -1)
out = cfg.path('raw')
translated.to_parquet(out / 'translated.parquet', index=False)
review_sheet(translated).to_csv(out / 'translated_review.csv', index=False)
translated[['text', 'attack_type']].head(10)

## 1.3–1.7 Генераторы, очистка, почти-дубли, разбиение

In [ ]:
import pandas as pd
from pig.data.build import build_dataset
from pig.features.embedder import make_embedder
emb = make_embedder(cfg, 'sentence', cfg.path('cache'))
tp = cfg.path('raw') / 'translated.parquet'
translated = pd.read_parquet(tp) if tp.exists() else None
df, stats = build_dataset(cfg, use_hf=True, embedder=emb, translated=translated, out_dir=cfg.path('processed'))
stats

## Проверки
Пересечений между выборками нет, в каждой выборке оба класса и оба языка, hard negatives есть в test. Ручной тест-сет (`manual`) проверяется отдельно.

In [ ]:
assert all(v == 0 for v in stats['leakage'].values()), stats['leakage']
for s, part in df[df['split'] != 'manual'].groupby('split'):
    assert set(part['label']) == {0, 1}, s
    assert set(part['lang']) == {'ru', 'en'}, s
assert df[df['split'] == 'test']['is_hard_negative'].any()
pd.crosstab([df['split'], df['lang']], df['label'], margins=True)

## Ручной тест-сет
Заполняется по инструкции `data/manual/README.md`. Пока файл пустой, выборки `manual` нет.

In [ ]:
manual = df[df['split'] == 'manual']
print('manual:', len(manual))
if len(manual):
    display(pd.crosstab(manual['attack_type'], manual['is_hard_negative']))

## 1.8 EDA

In [ ]:
from IPython.display import Image
from pig.eval.plots import plot_eda
for p in plot_eda(df, cfg.path('reports')):
    display(Image(str(p)))
pd.crosstab(df['attack_type'], df['split'])

In [ ]:
for t in ['override', 'leak', 'roleplay', 'obfuscation']:
    print('==', t)
    for x in df[df['attack_type'] == t].sample(3, random_state=0)['text']:
        print('  ', x[:160])